### Subnet Mask for /26 and Usable Hosts
A /26 means we have 26 bits reserved for the network portion, leaving 6 bits for hosts.

Calculation:
Network bits:  26
Host bits:     32 - 26 = 6
Total hosts:   2^6 = 64 addresses
Usable hosts:  64 - 2 = 62

The subnet mask is built by turning 26 bits ON, then the remaining 6 bits OFF:
11111111.11111111.11111111.11000000
= 255.255.255.192

Subnet mask: 255.255.255.192
Usable hosts: 62 per subnet

### 6 Subnets with at Least 25 Hosts Each
Starting network: 10.0.0.0/24. We need at least 6 subnets, each supporting 25+ usable hosts.

Step 1 — Figure out how many host bits we need for 25 hosts:
2^4 = 16   → not enough (only 14 usable)
2^5 = 32   → 30 usable hosts ✓  (covers 25+)
So we need 5 host bits, which means the prefix is:
32 - 5 = /27

Step 2 — Check how many subnets /27 gives us inside a /24:
Extra network bits borrowed: 27 - 24 = 3 bits
Number of subnets: 2^3 = 8 subnets  ✓  (covers the 6 we need)

Prefix length to use: /27  (gives 8 subnets × 30 usable hosts each)

### Network and Broadcast Address of 172.16.5.130/27
We need to find the network block that 172.16.5.130 falls into with a /27 mask.

The /27 subnet mask is 255.255.255.224. The interesting octet is the last one.
With /27, subnets increment by 32 in the last octet:
172.16.5.0    — 172.16.5.31
172.16.5.32   — 172.16.5.63
172.16.5.64   — 172.16.5.95
172.16.5.96   — 172.16.5.127
172.16.5.128  — 172.16.5.159  ← 130 falls here

Verification using binary AND with mask:
130 in binary:  10000010
224 in binary:  11100000
AND result:     10000000 = 128  → network address last octet

Network address:   172.16.5.128
Broadcast address: 172.16.5.159
Usable range:      172.16.5.129 – 172.16.5.158

### Why Do We Subtract 2 for Usable Hosts?
When you create a subnet, the first and last addresses are reserved and cannot be assigned to devices:

•	The first address is the Network Address — it identifies the subnet itself. Routers use this to know which network a packet belongs to. You cannot give this to a computer.
•	The last address is the Broadcast Address — any packet sent to this address goes to every device in the subnet. It is used by protocols like ARP. You cannot assign this to a device either.

So out of every block of addresses, two are always taken by these roles, and only the addresses in between are free for actual hosts — hence the formula:
Usable hosts = 2^(host bits) - 2

### CIDR vs. Classful Addressing
In the early days of networking, IP addresses were divided into fixed classes:

•	Class A — first octet 1-126, default mask /8 (16 million hosts). Way too big for most organisations.
•	Class B — first octet 128-191, default mask /16 (65,534 hosts). Still very large.
•	Class C — first octet 192-223, default mask /24 (254 hosts). Often too small.

The problem with classful addressing is that it is rigid. A company needing 300 hosts had to get a Class B (wasting over 65,000 addresses). This was causing the IPv4 address space to run out fast.

CIDR — Classless Inter-Domain Routing — was introduced to fix this. Instead of fixed classes, CIDR lets you set the prefix length to exactly what you need using slash notation (e.g. /26, /19). This means:

•	Addresses are allocated more efficiently — no more huge blocks handed to small networks.
•	Route summarisation (supernetting) becomes possible, which shrinks routing tables.
•	You can split and combine blocks freely based on actual requirements.

In short: classful addressing wastes space with rigid blocks, while CIDR gives you full control over how big or small a network block is.

### Subnet Size for a 2-IP Router Link
A point-to-point link between two routers only needs exactly 2 usable IP addresses — one for each router's interface. We want the smallest subnet that still gives us 2 usable hosts.

Working backwards from the formula:
2^(host bits) - 2 >= 2
2^2 - 2 = 2  ✓
So we only need 2 host bits, giving us a /30 subnet:
Total addresses: 4
Network address: 1 (reserved)
Broadcast:       1 (reserved)
Usable hosts:    2  ✓

Subnet to use: /30  (e.g. 10.0.0.0/30 — gives .1 and .2 for the two routers)

Note: Some engineers use /31 (RFC 3021) which skips the network/broadcast concept entirely and gives exactly 2 addresses for point-to-point links, but /30 is the traditional and most widely supported choice.

### VLSM
VLSM stands for Variable Length Subnet Mask.

In basic subnetting you take one network and split it into equal-sized pieces. That is fine for simple setups, but real networks have very different needs — one segment might need 100 hosts, another only needs 10, and a router link needs just 2. Forcing them all into the same size subnet wastes a lot of addresses.

VLSM solves this by letting you use different prefix lengths on different subnets carved out of the same address space. You can subnet your subnets — breaking a block into unequal pieces sized to fit each segment exactly.

Why it matters:
•	Efficient use of IP space — you only allocate what each segment actually needs.
•	Works hand-in-hand with CIDR, since both rely on classless prefix notation.
•	Essential for real-world network design where different links and segments have wildly different host requirements.

Example: Starting with 192.168.1.0/24, you could assign /25 to a large LAN (126 hosts), /27 to a smaller segment (30 hosts), and /30 to each router link (2 hosts) — all from the same parent block without overlap.


